In [0]:
%sql

SELECT current_catalog() AS catalog_name, current_schema() AS schema_name;


In [0]:
%sql
CREATE OR REPLACE TABLE
    `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab
USING DELTA
AS
SELECT
    application_id,
    CAST(loan_amount AS DECIMAL(12,2)) AS loan_amount,
    action_taken
FROM VALUES
    ('A101', 250000, 'originated'),
    ('A102', 180000, 'denied'),
    ('A103', 320000, 'originated')
AS t(application_id, loan_amount, action_taken);

In [0]:
%sql

SELECT *
FROM `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab;

In [0]:
%sql

DESCRIBE DETAIL `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab;


In [0]:
%sql

DESCRIBE HISTORY `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab;


In [0]:
%sql
-- introduce duplicates

INSERT INTO `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab
VALUES ('A104', 275000.00, 'originated');


In [0]:
%sql

SELECT COUNT(*) AS total_records
FROM `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab;


In [0]:
%sql

SELECT
    application_id,
    COUNT(*) AS occurrence_count
FROM `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab
GROUP BY application_id
HAVING COUNT(*) > 1;


In [0]:
%sql
select * FROM `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab
version as of 2

In [0]:
%sql

CREATE OR REPLACE TABLE `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_merge_lab
USING DELTA
AS
SELECT *
FROM `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_delta_lab
VERSION AS OF 0;


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW mortgage_batch AS
SELECT
    application_id,
    CAST(loan_amount AS DECIMAL(12,2)) AS loan_amount,
    action_taken
FROM VALUES
    ('A102', 185000.00, 'denied'),
    ('A104', 275000.00, 'originated')
AS source(application_id, loan_amount, action_taken);


In [0]:
%sql

MERGE INTO `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_merge_lab AS target
USING mortgage_batch AS source
ON target.application_id = source.application_id

WHEN MATCHED AND (
    NOT (target.loan_amount <=> source.loan_amount)
    OR NOT (target.action_taken <=> source.action_taken)
)
THEN UPDATE SET
    target.loan_amount = source.loan_amount,
    target.action_taken = source.action_taken

WHEN NOT MATCHED
THEN INSERT (
    application_id,
    loan_amount,
    action_taken
)
VALUES (
    source.application_id,
    source.loan_amount,
    source.action_taken
);


In [0]:
%sql

SELECT *
FROM `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_merge_lab
ORDER BY application_id;


In [0]:
%sql

DESCRIBE HISTORY `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_merge_lab


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW nightly_snapshot AS
SELECT
    application_id,
    CAST(loan_amount AS DECIMAL(12,2)) AS loan_amount,
    action_taken
FROM VALUES
    ('A101', 450000, 'originated'),
    ('A103', 625000, 'originated')
AS s(application_id, loan_amount, action_taken);


In [0]:
%sql

INSERT INTO TABLE `mortgage-lending-risk-and-regulatory-analytics`.default.mortgage_merge_lab as target
REPLACE on target.application_id = source.application_id
(SELECT * FROM nightly_snapshot) as source;